In [ ]:
import numpy as np
import plotly.graph_objects as go
from myst_nb import glue
from IPython.display import HTML

ZGAP = 1.5  # depth gap between batch slices (y-axis in 3D space)
GAP = 2.0  # horizontal gap between matrices
SERIF = 0.22  # length of the matrix-bracket feet
BRACKET_W = 5  # bracket line thickness (device px)
NUM_SIZE = 14  # matrix-element font size
FADE = 0.4  # how much the rear-most batch slice fades toward white (0..1)
PLOT_HEIGHT = 250  # height of the plot
BRACKET_OFFSET = -0.3  # shift brackets to align with numbers under 3D perspective
EXTRA_PLANES = 2  # number of ghost planes drawn behind the front slice

VIEW = dict(x=0, y=-1.7, z=0)

# ---- example tensors:  B=1, T=3, d_k=4, d_v=2 -------------------------------
Q = np.array([
    [[1, 0, 2, 1], [0, 1, 1, 0], [2, 1, 0, 1]],
])  # (B, T, d_k)
K = np.array([
    [[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]],
])  # (B, T, d_k)
V = np.array([[[1, 2], [3, 4], [5, 6]]])  # (B, T, d_v)

d_k = Q.shape[-1]
KT = K.transpose(0, 2, 1)  # (B, d_k, T)
scores = np.matmul(Q, KT)  # (B, T, T)
scaled = scores / np.sqrt(d_k)  # (B, T, T)
_e = np.exp(scaled - scaled.max(axis=-1, keepdims=True))
weights = _e / _e.sum(axis=-1, keepdims=True)  # (B, T, T)
output = np.matmul(weights, V)  # (B, T, d_v)

# colors per quantity
C_Q, C_K, C_S = "#2C5C8A", "#4F7A1A", "#A86A12"
C_SCALED, C_W, C_V, C_O = "#B5651D", "#7A4FA0", "#1F6F8B", "#1F7A6B"


def lighten(hex_color, t):
    """Blend a #rrggbb color toward white by fraction t (0 = unchanged, 1 = white)."""
    r, g, b = (int(hex_color[i : i + 2], 16) for i in (1, 3, 5))
    r, g, b = (round(c + (255 - c) * t) for c in (r, g, b))
    return f"#{r:02x}{g:02x}{b:02x}"


def _draw_plane(fig, x_lo, x_hi, z_lo, z_hi, y, color):
    fig.add_trace(
        go.Mesh3d(
            x=[x_lo + 0.05, x_hi - 0.05, x_hi - 0.05, x_lo + 0.05],
            y=[y, y, y, y],
            z=[z_lo + 0.05, z_lo + 0.05, z_hi - 0.05, z_hi - 0.05],
            i=[0, 0],
            j=[1, 2],
            k=[2, 3],
            color=color,
            opacity=0.2,
            hoverinfo="none",
            showlegend=False,
        )
    )


def add_matrix(fig, mat, x0, color, name, shape):
    """Render a tensor as 3D numbers + brackets on the front slice only.
    EXTRA_PLANES ghost planes are drawn behind it. Returns (x_lo, x_hi)."""
    _, rows, cols = mat.shape
    decimals = 0 if np.allclose(mat, np.round(mat)) else 2
    x_lo, x_hi = x0 - 0.5, x0 + cols - 0.5
    z_lo, z_hi = -0.5 - BRACKET_OFFSET, rows - 0.5 - BRACKET_OFFSET
    total = 1 + EXTRA_PLANES
    for b in range(total):
        y = b * ZGAP
        t = FADE * (b / (total - 1)) if total > 1 else 0.0
        c = lighten(color, t)
        if b == 0:
            # front slice: numbers + brackets
            tx, ty, tz, txt = [], [], [], []
            for r in range(rows):
                for col in range(cols):
                    tx.append(x0 + col)
                    ty.append(y)
                    tz.append(rows - 1 - r)
                    txt.append(f"<b>{mat[0, r, col]:.{decimals}f}</b>")
            left = [
                (x_lo + SERIF, z_hi),
                (x_lo, z_hi),
                (x_lo, z_lo),
                (x_lo + SERIF, z_lo),
            ]
            right = [
                (x_hi - SERIF, z_hi),
                (x_hi, z_hi),
                (x_hi, z_lo),
                (x_hi - SERIF, z_lo),
            ]
            bx, by, bz = [], [], []
            for seq in (left, right):
                for px, pz in seq:
                    bx.append(px)
                    by.append(y)
                    bz.append(pz)
                bx.append(None)
                by.append(None)
                bz.append(None)
            fig.add_trace(
                go.Scatter3d(
                    x=bx,
                    y=by,
                    z=bz,
                    mode="lines",
                    line=dict(color=c, width=BRACKET_W),
                    hoverinfo="none",
                    showlegend=False,
                    opacity=0.85,
                )
            )
            fig.add_trace(
                go.Scatter3d(
                    x=tx,
                    y=ty,
                    z=tz,
                    mode="text",
                    text=txt,
                    textfont=dict(size=NUM_SIZE, color=c),
                    hoverinfo="none",
                    showlegend=False,
                )
            )
        else:
            _draw_plane(fig, x_lo, x_hi, z_lo, z_hi, y, c)

    xc = x0 + (cols - 1) / 2
    fig.add_trace(
        go.Scatter3d(
            x=[xc],
            y=[0],
            z=[rows + 0.45],
            mode="text",
            text=[f"<b>{name}</b>"],
            textfont=dict(size=16, color=color),
            hoverinfo="none",
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=[xc],
            y=[0],
            z=[rows - 0.05],
            mode="text",
            text=[shape],
            textfont=dict(size=11, color=color),
            hoverinfo="none",
            showlegend=False,
        )
    )
    return x_lo, x_hi


def compose(specs, ops=None, height=400):
    """specs: list of (mat, color, name, shape). ops: operator strings between them.
    Returns a self-contained HTML object ready to glue."""
    ops = ops or []
    fig = go.Figure()
    x0, bounds = 0.0, []
    for mat, color, name, shape in specs:
        bounds.append(add_matrix(fig, mat, x0, color, name, shape))
        x0 += mat.shape[2] + GAP
    for i, op in enumerate(ops):
        xmid = (bounds[i][1] + bounds[i + 1][0]) / 2
        fig.add_trace(
            go.Scatter3d(
                x=[xmid],
                y=[0],
                z=[1.0],
                mode="text",
                text=[f"<b>{op}</b>"],
                textfont=dict(size=24 if len(op) == 1 else 14, color="#333"),
                hoverinfo="none",
                showlegend=False,
            )
        )
    fig.update_layout(
        template="plotly_white",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data",
            camera=dict(eye=VIEW),
        ),
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0),
        height=height,
    )
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


In [ ]:
# Step 1 -- scores = Q @ Kᵀ  (one T×T score matrix per batch element)
glue(
    "scores_plot",
    compose(
        [
            (Q, C_Q, "Q", "(B, T, d<sub>k</sub>)"),
            (KT, C_K, "K<sup>T</sup>", "(B, d<sub>k</sub>, T)"),
            (scores, C_S, "scores", "(B, T, T)"),
        ],
        ops=["@", "="],
        height=PLOT_HEIGHT,
    ),
    display=False,
)

In [ ]:
# Step 2 -- scale by 1/sqrt(d_k)  (here √d_k = 2)
glue(
    "scaled_plot",
    compose(
        [
            (scores, C_S, "QK<sup>T</sup>", "(B, T, T)"),
            (scaled, C_SCALED, "scaled", "(B, T, T)"),
        ],
        ops=["÷ √dₖ"],
        height=PLOT_HEIGHT,
    ),
    display=False,
)

In [ ]:
# Step 3 -- row-wise softmax  (each row sums to 1)
glue(
    "weights_plot",
    compose(
        [
            (scaled, C_SCALED, "scaled", "(B, T, T)"),
            (weights, C_W, "W", "(B, T, T)"),
        ],
        ops=["softmax"],
        height=PLOT_HEIGHT,
    ),
    display=False,
)

In [ ]:
# Step 4 -- output = W @ V
glue(
    "output_plot",
    compose(
        [
            (weights, C_W, "W", "(B, T, T)"),
            (V, C_V, "V", "(B, T, d<sub>v</sub>)"),
            (output, C_O, "Output", "(B, T, d<sub>v</sub>)"),
        ],
        ops=["@", "="],
        height=PLOT_HEIGHT,
    ),
    display=False,
)